## Post Process the CSV Files from GPS Example C++ 

In [219]:
import pandas as pd
import pylupnt as pnt

In [ ]:
# Load the data
filter_name = "EKF"
use_gps = True
use_galileo = True
use_qzss = True
clock = "MiniRafs"  # "MiniRafs" or "Csac"

name = []
if use_gps:
    name.append("GPS")
if use_galileo:
    name.append("_GALILEO")
if use_qzss:
    name.append("_QZSS")


data_dir = pnt.LUPNT_DATA_PATH + "/../output/Example{}_{}/{}".format(filter_name, "".join(name), clock)
print(data_dir)

rv_true = 1000 * pd.read_csv(data_dir + "/rv.csv").values[:, 1:]
rv_est = 1000 * pd.read_csv(data_dir + "/rv_est.csv").values[:, 1:]
clk_true = 3e8 * pd.read_csv(data_dir + "/clk.csv").values[:, 1:]
clk_est = 3e8 * pd.read_csv(data_dir + "/clk_est.csv").values[:, 1:]
P_rv = 1e6 * pd.read_csv(data_dir + "/P_rv.csv").values[:, 1:]
P_clk = 9e16 * pd.read_csv(data_dir + "/P_clk.csv").values[:, 1:]
tspan = pd.read_csv(data_dir + "/rv.csv").values[:, 0]

tspan = (tspan - tspan[0]).flatten()/3600

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def plot_est_error_with_cov(tspan, est_error, P, ax, ylabel, ymax=None):
    ax.plot(tspan, est_error, label="Estimation Error")
    ax.fill_between(tspan, - 3 * np.sqrt(P), + 3 * np.sqrt(P), alpha=0.2, label="3-sigma")
    ax.set_xlabel("Time [hr]")
    ax.set_ylabel(ylabel)
    ax.grid()
    if ymax is not None:
        ax.set_ylim([-ymax, ymax])
    ax.set_xlim([tspan[0], tspan[-1]])
    ax.legend()

fig, ax = plt.subplots(2, 4, figsize=(14, 6))
labels = ['x [m]', 'y [m]', 'z [m]', 'vx [mm/s]', 'vy [mm/s]', 'vz [mm/s]', 'Clock Bias [m]', 'Clock Drift [m/s]']
for i in range(3):
    plot_est_error_with_cov(tspan, rv_est[:, i] - rv_true[:, i], P_rv[:, i], ax[0, i], labels[i], ymax=500)
    plot_est_error_with_cov(tspan, 1000 * (rv_est[:, i + 3] - rv_true[:, i + 3]), 1e6 * P_rv[:, i + 3], ax[1, i], labels[i+3], ymax=200.0)

plot_est_error_with_cov(tspan, clk_est[:, 0] - clk_true[:, 0], P_clk[:, 0], ax[0, 3], labels[6], ymax=1000)

# clock drift0
plot_est_error_with_cov(tspan, 1000 * (clk_est[:, 1] - clk_true[:, 1]), 1e6 * P_clk[:, 1], ax[1, 3], labels[7], ymax=100)

plt.tight_layout()
plt.savefig(data_dir + "/error.pdf")
plt.show()


## Convert to RTN Frame

In [ ]:
print("Sim Length: {} hr".format(tspan[-1]))

In [ ]:
import os

def cart_to_rtn(rv_ref, rv):
    r = rv[0:3] - rv_ref[0:3]
    v = rv[3:6] - rv_ref[3:6]
    i = r / np.linalg.norm(r)
    n = np.cross(r, v)
    k = n / np.linalg.norm(n)
    j = np.cross(k, i)
    R = np.array([i, j, k]).T
    r_rtn = np.dot(R, r)
    v_rtn = np.dot(R, v)
    return np.hstack([r_rtn, v_rtn]), R


error_rtn = np.zeros((len(tspan), 6))
P_rtn = np.zeros((len(tspan), 6))
for ti in range(len(tspan)):
    error_rtn[ti, :], Rmat = cart_to_rtn(rv_true[ti, :], rv_est[ti, :])
    Rmat6 = np.block([[Rmat, np.zeros((3, 3))], [np.zeros((3, 3)), Rmat]])
    P_rtn[ti, :] = np.diag(Rmat6 @ np.diag(P_rv[ti, :]) @ Rmat6.T)


labels = ['Radial Position [m]', 'Tangential Position [m]', 'Normal Position [m]', 'Radial Velocity [mm/s]', 'Tangential Velocity [mm/s]', 'Normal Velocity [mm/s]', 'Clock Bias [m]', 'Clock Drift [m/s]']

fig, ax = plt.subplots(2, 4, figsize=(14, 6))
for i in range(3):
    plot_est_error_with_cov(tspan, error_rtn[:, i], P_rtn[:, i], ax[0, i], labels[i], ymax=200)
    plot_est_error_with_cov(tspan, 1000 * error_rtn[:, i + 3], 1e6 * P_rtn[:, i + 3], ax[1, i], labels[i+3], ymax=100.0)

plot_est_error_with_cov(tspan, clk_est[:, 0] - clk_true[:, 0], P_clk[:, 0], ax[0, 3], labels[6], ymax=100)

# clock drift0
plot_est_error_with_cov(tspan, 1000 * (clk_est[:, 1] - clk_true[:, 1]), 1e6 * P_clk[:, 1], ax[1, 3], labels[7], ymax=50)

plt.tight_layout()

# save as pdf
outdir = "output/ex_{}_gnss".format(filter_name)
os.makedirs(outdir, exist_ok=True)
plt.savefig(outdir + "/ekf_rtn.pdf")
plt.show()